# Indoone AI GPU Training (Google Colab)

Run Indoone model training on a Colab GPU instead of GitHub-hosted CPU.  
This notebook follows the repo training pipeline, runs validation + behavioral gate, then optionally promotes/uploads the model.


In [ ]:
# 1) Clone the exact training repo
import os, subprocess, sys, json, pathlib

REPO = "https://github.com/indooneteam/Indoone-Backend.git"
BRANCH = "main"
WORKDIR = "/content/Indoone-Backend"

if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
print("repo:", os.getcwd())
print("commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
# 2) Prepare Colab dependencies without replacing Colab's working Torch unnecessarily
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "fastapi>=0.115,<1.0",
    "uvicorn[standard]>=0.34,<1.0",
    "pydantic>=2.0,<3.0",
    "httpx>=0.27,<1.0",
    "cryptography>=44,<46",
    "pytest>=8.0,<9.0",
    "pytest-asyncio>=0.26,<1.0",
    "numpy>=1.26,<3.0",
    "boto3>=1.35,<2.0",
    "pypdf>=6.0,<7.0",
    "python-docx>=1.2,<2.0",
    "openpyxl>=3.1,<4.0",
], check=True)

import torch
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is not active. In Colab choose Runtime -> Change runtime type -> GPU, then rerun.")

print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 3) Build and validate the training dataset
import subprocess, sys

def run(*args):
    print("$", " ".join(args), flush=True)
    subprocess.run(list(args), check=True)

run(sys.executable, "scripts/build_multilingual_training_pack.py")
run(sys.executable, "scripts/prepare_dataset.py",
    "--source", "data/raw/indoone_corpus.txt",
    "--output-dir", "data/processed")
run(sys.executable, "scripts/assemble_training_dataset.py",
    "--output", "data/processed/instructions_train.jsonl",
    "--validation-output", "data/processed/instructions_validation.jsonl")
run(sys.executable, "scripts/validate_dataset_quality.py",
    "--source", "data/processed/instructions_train.jsonl",
    "--source", "data/processed/instructions_validation.jsonl")
run(sys.executable, "scripts/validate_training_manifest.py")
print("Dataset preparation + validation: OK")


In [ ]:
# 4) GPU training
# GPU budget is intentionally stronger than the hosted CPU fallback.
# The training loop now mixes instruction-focused batches with general corpus batches.
run(
    sys.executable, "scripts/run_training_pipeline.py",
    "--steps", "8000",
    "--batch-size", "16",
    "--checkpoint-interval", "500",
    "--learning-rate", "3e-4",
    "--seed", "42",
    "--instruction-mix-ratio", "0.7",
    "--skip-upload",
)
print("Training + base evaluation: OK")


In [ ]:
# 5) Behavioral promotion gate
run(
    sys.executable, "-m", "app.ai.behavior_eval",
    "--cases", "data/eval/behavior.jsonl",
    "--checkpoint", "models/indoone-small/indoone-small.pt",
    "--tokenizer", "models/indoone-small/tokenizer.json",
    "--output", "models/indoone-small/behavior_eval.json",
)
print("Behavior gate: OK")


In [ ]:
# 6) Response diversity smoke check
# Catch the specific regression where unrelated questions receive the same reply.
from app.ai.inference import LocalModelRuntime

smoke_prompts = [
    "What is 12 + 7?",
    "Explain what an API is in simple words.",
    "Give me one practical study tip.",
    "Translate 'Good morning' into Kannada.",
    "What should you do when you do not know an answer?",
    "Why is training data quality important for an AI model?",
]
runtime = LocalModelRuntime(
    pathlib.Path("models/indoone-small/indoone-small.pt"),
    pathlib.Path("models/indoone-small/tokenizer.json"),
)
answers = [runtime.generate(prompt, max_new_tokens=80, temperature=0.0) for prompt in smoke_prompts]
for prompt, answer in zip(smoke_prompts, answers):
    print(f"USER: {prompt}\nINDOONE: {answer}\n")
normalized = [" ".join(answer.casefold().split())[:240] for answer in answers]
unique_count = len(set(normalized))
max_duplicate_count = max(normalized.count(item) for item in set(normalized))
print(f"unique_answer_prefixes={unique_count}/{len(answers)}")
print(f"largest_duplicate_group={max_duplicate_count}")
if unique_count < 4 or max_duplicate_count > 3:
    raise RuntimeError("Response diversity check failed: unrelated prompts are receiving overly similar answers.")
print("Response diversity smoke check: OK")


In [ ]:
# 7) Promote locally; B2 upload is optional and uses Colab Secrets (never paste credentials here).
import os, json, subprocess, sys

version = "colab-" + subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
run(
    sys.executable, "scripts/promote_model.py",
    "--version", version,
    "--model-dir", "models/indoone-small",
    "--behavior-cases", "data/eval/behavior.jsonl",
    "--registry", "models/registry.json",
)
print("Local promotion: OK")
print("model version:", version)


## 8) Optional: publish to Backblaze B2

Set these in **Colab -> Secrets** (not in notebook code):
`B2_APPLICATION_KEY_ID`, `B2_APPLICATION_KEY`, `B2_BUCKET_NAME`, `B2_ENDPOINT`.

Then run the next cell.


In [ ]:
# 8) Optional B2 publish
from google.colab import userdata
import os

secret_names = [
    "B2_APPLICATION_KEY_ID",
    "B2_APPLICATION_KEY",
    "B2_BUCKET_NAME",
    "B2_ENDPOINT",
]
for name in secret_names:
    os.environ[name] = userdata.get(name)

run(
    sys.executable, "scripts/upload_model_to_b2.py",
    "--model-dir", "models/indoone-small",
    "--prefix", "models/indoone-small",
)
print("B2 upload: OK")


In [ ]:
# 9) Final artifact check
from pathlib import Path
for name in [
    "models/indoone-small/indoone-small.pt",
    "models/indoone-small/tokenizer.json",
    "models/indoone-small/metadata.json",
    "models/indoone-small/training_history.json",
    "models/indoone-small/behavior_eval.json",
    "models/registry.json",
]:
    path = Path(name)
    print(f"{name}: {'OK' if path.is_file() else 'MISSING'}", path.stat().st_size if path.is_file() else "")
